In [ ]:
print("STEP 1: Data Cleaning → Genera df_final.parquet")

import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
base_path = Path(r"C:/Users/rroman/OneDrive - Empresa Eléctrica de Guatemala, S.A. (EEGSA)/Documents/Unidad/2026/Capacitación/Data Science y Machine Learning/proyecto final/data")

raw_path = base_path / "raw"
interim_path = base_path / "interim"
interim_path.mkdir(parents=True, exist_ok=True)

files = list(raw_path.glob("intervals_0000_*.txt"))
print("Archivos encontrados:", len(files))

In [14]:
df_list = []

columnas = [
    "Record Type",
    "Record Version",
    "Time Stamp",
    "Premise ID",
    "ESIID",
    "Provisioned",
    "Meter ID",
    "Purpose",
    "Commodity",
    "Units",
    "Calculation Constant",
    "Interval",
    "Count",
    "FirstIntervalDateTime",
    "Data"
]

for file in files:
    print("Procesando:", file)
    
    with open(file, 'r', encoding='latin-1') as f:
        rows = []
        
        for line in f:
            parts = line.strip().split("~")
            
            fixed = parts[:14]                 # columnas fijas
            data = "~".join(parts[14:])       # intervalos
            
            rows.append(fixed + [data])
    
    temp = pd.DataFrame(rows, columns=columnas)
    df_list.append(temp)

df = pd.concat(df_list, ignore_index=True)

print("Shape:", df.shape)
print(df.head())

Procesando: C:\Users\rroman\OneDrive - Empresa Eléctrica de Guatemala, S.A. (EEGSA)\Documents\Unidad\2026\Capacitación\Data Science y Machine Learning\proyecto final\data\raw\intervals_0000_01032026.txt
Procesando: C:\Users\rroman\OneDrive - Empresa Eléctrica de Guatemala, S.A. (EEGSA)\Documents\Unidad\2026\Capacitación\Data Science y Machine Learning\proyecto final\data\raw\intervals_0000_02032026.txt
Procesando: C:\Users\rroman\OneDrive - Empresa Eléctrica de Guatemala, S.A. (EEGSA)\Documents\Unidad\2026\Capacitación\Data Science y Machine Learning\proyecto final\data\raw\intervals_0000_03032026.txt
Procesando: C:\Users\rroman\OneDrive - Empresa Eléctrica de Guatemala, S.A. (EEGSA)\Documents\Unidad\2026\Capacitación\Data Science y Machine Learning\proyecto final\data\raw\intervals_0000_04032026.txt
Procesando: C:\Users\rroman\OneDrive - Empresa Eléctrica de Guatemala, S.A. (EEGSA)\Documents\Unidad\2026\Capacitación\Data Science y Machine Learning\proyecto final\data\raw\intervals_000

In [16]:
df = df[df["Record Type"] != "Record Type"]
df = df[df["ESIID"].isin(["41-0330", "41-0331"])]

print("Después de filtrar ESIID:", df.shape)

Después de filtrar ESIID: (1434844, 15)


In [17]:
normalizacion = {
    # Energía
    "KWH": "kWh_Del",
    "kWh-DelLPABC": "kWh_Del",
    
    "-KWH": "kWh_Rec",
    "kWh-RcvdLPABC": "kWh_Rec",
    
    "kVARhDel": "kVARh_Del",
    "kVARh-DelLPABC": "kVARh_Del",
    
    "kVARh-Rcvd": "kVARh_Rec",
    "kVARh-RcvdLPABC": "kVARh_Rec",
    
    # Corriente
    "CalculatedIrmsPhaseALP": "CurrentPhaseA",
    "CalculatedIrmsPhaseALPABC": "CurrentPhaseA",
    "IrmsPhaseALPABC": "CurrentPhaseA",
    
    "CalculatedIrmsPhaseBLP": "CurrentPhaseB",
    "CalculatedIrmsPhaseBLPABC": "CurrentPhaseB",
    "IrmsPhaseBLPABC": "CurrentPhaseB",
    
    "CalculatedIrmsPhaseCLP": "CurrentPhaseC",
    "CalculatedIrmsPhaseCLPABC": "CurrentPhaseC",
    "IrmsPhaseCLPABC": "CurrentPhaseC",
    
    # Voltaje
    "CalculatedVrmsPhaseALP": "VoltagePhaseA",
    "CalculatedVrmsPhaseALPABC": "VoltagePhaseA",
    "VrmsPhaseALPABC": "VoltagePhaseA",
    
    "CalculatedVrmsPhaseBLP": "VoltagePhaseB",
    "CalculatedVrmsPhaseBLPABC": "VoltagePhaseB",
    "VrmsPhaseBLPABC": "VoltagePhaseB",
    
    "CalculatedVrmsPhaseCLP": "VoltagePhaseC",
    "CalculatedVrmsPhaseCLPABC": "VoltagePhaseC",
    "VrmsPhaseCLPABC": "VoltagePhaseC",
    
    # Eventos
    "Sag_V_Any_Phase": "VoltageSag",
    "VsagAnyPhaseLPABC": "VoltageSag",
    
    "Swell_V_Any_Phase": "VoltageSwell",
    "VswellAnyPhaseLPABC": "VoltageSwell",
    
    # Neutro
    "CalculatedIrmsNeutralLP": "CurrentPhaseN",
    "CalculatedIrmsNeutralLPABC": "CurrentPhaseN",
    "IrmsNeutralLPABC": "CurrentPhaseN"
}

df["Units"] = df["Units"].str.strip().replace(normalizacion)

In [18]:
print(df.head(20))

      Record Type Record Version        Time Stamp  Premise ID    ESIID  \
23346     MEPMD01       20080519  03012026120500AM  0002139303  41-0330   
23347     MEPMD01       20080519  03012026120500AM  0002139303  41-0330   
23348     MEPMD01       20080519  03012026120500AM  0002139303  41-0330   
23349     MEPMD01       20080519  03012026120500AM  0002139303  41-0330   
23350     MEPMD01       20080519  03012026120500AM  0002139303  41-0330   
23351     MEPMD01       20080519  03012026120500AM  0002139303  41-0330   
23352     MEPMD01       20080519  03012026120500AM  0002139303  41-0330   
23353     MEPMD01       20080519  03012026120500AM  0002139303  41-0330   
23354     MEPMD01       20080519  03012026120500AM  0002139303  41-0330   
23355     MEPMD01       20080519  03012026120500AM  0002139303  41-0330   
23356     MEPMD01       20080519  03012026120500AM  0002139303  41-0330   
23357     MEPMD01       20080519  03012026120500AM  0002139303  41-0330   
23358     MEPMD01       2

In [19]:
output_file = interim_path / "df_final.parquet"

df.to_parquet(output_file, engine="pyarrow", index=False)

print("Guardado:", output_file)

Guardado: C:\Users\rroman\OneDrive - Empresa Eléctrica de Guatemala, S.A. (EEGSA)\Documents\Unidad\2026\Capacitación\Data Science y Machine Learning\proyecto final\data\interim\df_final.parquet
